# Benchmark reproduction entry point

This notebook separates published-summary analysis from recomputation using keyed predictions. It does not run any model. CPU only, Python 3.10+. See `docs/BENCHMARK_RELEASE.md` for inference and release prerequisites. A report with `complete: false` is not a successful full reproduction.


In [ ]:
from pathlib import Path
import json, subprocess, sys, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/benchmark_release.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook inside the release source directory')
OUT = Path(tempfile.mkdtemp(prefix='eva-benchmark-', dir=ROOT))
print('Outputs:', OUT)


## A. Published summaries to diagnostic plots

Uses the bundled, hashed CSV snapshots. These are final per-assay metrics, not per-sequence predictions. Expected model scope comes from the existing plotting code. The archived protein table lacks Evo2 7B; exit status 2 and a missing-results marker are therefore expected, not suppressed. The generated plots are diagnostic summaries, not replacement manuscript artwork.


In [ ]:
command = [sys.executable, str(ROOT / 'scripts/benchmark_release.py'), 'audit-reference', '--manifest', str(ROOT / 'reproduction/benchmark_release/reference_manifest.json'), '--data-root', str(ROOT / 'reproduction/benchmark_release/reference'), '--output', str(OUT / 'reference'), '--plot']
run = subprocess.run(command, capture_output=True, text=True)
print(run.stdout)
if run.returncode not in (0, 2):
    raise RuntimeError(run.stderr)
report = json.loads((OUT / 'reference/report.json').read_text())
print('Complete coverage:', report['complete'])
for group in report['groups']:
    print(group['id'], group['observed_pairs'], '/', group['expected_pairs'])
    print('Missing:', group['missing_pairs'])


## B. Keyed predictions to metrics

After preparing label/prediction CSVs and verified hashes, select a manifest below. Every model must cover the same dataset list. Rows are joined by exact variant ID and checked sequence, never by row position. Missing/duplicate rows and NaN/Inf are errors. The scorer reports signed Spearman and mean absolute Spearman; it never flips score signs to match a table.

The supplied template is intentionally incomplete and cannot be used to claim a paper run. A successful arithmetic comparison does not verify checkpoint lineage or biological label provenance.


In [ ]:
EVALUATION_MANIFEST = None  # Set to a completed, validated manifest Path.
EVALUATION_DATA_ROOT = None
if EVALUATION_MANIFEST is None:
    print('Prediction recomputation NOT RUN: supply a completed manifest and data root.')
else:
    if EVALUATION_DATA_ROOT is None:
        raise ValueError('Set EVALUATION_DATA_ROOT explicitly')
    result = subprocess.run([sys.executable, str(ROOT / 'scripts/benchmark_release.py'), 'evaluate', '--manifest', str(EVALUATION_MANIFEST), '--data-root', str(EVALUATION_DATA_ROOT), '--output', str(OUT / 'predictions'), '--plot'], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError('Evaluation incomplete, unresolved, or mismatched; inspect report.json. ' + result.stderr)


## C. Fresh EVA and competitor inference

For the frozen EVA-1.4B × Milena workflow, follow `docs/REPRODUCTION.md` and run `scripts/reproduce_milena.py`. This notebook does not execute GPU inference. The workflow index lists competitor implementations and missing exact resource bindings.
